<a href="https://colab.research.google.com/github/ghada-dahdoh/Applied-natural-language-processing/blob/main/Lab_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers pandas matplotlib


In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

In [ ]:
data = {
   "language": ["ar", "ar", "ar", "en", "en", "en"],
   "text": [
       "احببت التجربه",
       "لم يرضني العمل",
       "اريد المشاركه",
       "I enjoyed the experince",
       "the work did not satisfy me",
       "I would like to participate"
   ]
}
df = pd.DataFrame(data)
df

,language,text
0,ar,احببت التجربه
1,ar,لم يرضني العمل
2,ar,اريد المشاركه
3,en,I enjoyed the experince
4,en,the work did not satisfy me
5,en,I would like to participate


In [ ]:
def normalize(text):
   text = str(text)
   text = re.sub(r"\s+", " ", text)
   return text.strip()

df["clean_text"] = df["text"].apply(normalize)
df[["text", "clean_text"]]

,text,clean_text
0,احببت التجربه,احببت التجربه
1,لم يرضني العمل,لم يرضني العمل
2,اريد المشاركه,اريد المشاركه
3,I enjoyed the experince,I enjoyed the experince
4,the work did not satisfy me,the work did not satisfy me
5,I would like to participate,I would like to participate


In [ ]:
df.loc[0, "clean_text"] = " اريد المشاركه تواصل على 0551234567 أو email@example.com"
df.loc[1, "clean_text"] = "لم يرضني العمل راسلني على gg@gmail.com أو 0509876543"
df[["clean_text"]]

def mask_pii(text):
   text = str(text)
   text = re.sub(r'\S+@\S+', '[EMAIL]', text)
   text = re.sub(r'05\d{8}', '[PHONE]', text)
   return text

df["masked_text"] = df["clean_text"].apply(mask_pii)
df[["clean_text", "masked_text"]]

,clean_text,masked_text
0,اريد المشاركه تواصل على 0551234567 أو email@e...,اريد المشاركه تواصل على [PHONE] أو [EMAIL]
1,لم يرضني العمل راسلني على gg@gmail.com أو 0509...,لم يرضني العمل راسلني على [EMAIL] أو [PHONE]
2,اريد المشاركه,اريد المشاركه
3,I enjoyed the experince,I enjoyed the experince
4,the work did not satisfy me,the work did not satisfy me
5,I would like to participate,I would like to participate


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
   "distilbert-base-multilingual-cased"
)

df["tokens"] = df["masked_text"].apply(
   lambda x: tokenizer.tokenize(x)
)

df[["masked_text", "tokens"]]

,masked_text,tokens
0,اريد المشاركه تواصل على [PHONE] أو [EMAIL],"[ا, ##ريد, ال, ##م, ##شارك, ##ه, تو, ##اصل, عل..."
1,لم يرضني العمل راسلني على [EMAIL] أو [PHONE],"[لم, ي, ##رض, ##ني, العمل, را, ##سل, ##ني, على..."
2,اريد المشاركه,"[ا, ##ريد, ال, ##م, ##شارك, ##ه]"
3,I enjoyed the experince,"[I, enjoyed, the, ex, ##peri, ##nce]"
4,the work did not satisfy me,"[the, work, did, not, sat, ##is, ##fy, me]"
5,I would like to participate,"[I, would, like, to, participate]"


In [ ]:
df["token_count"] = df["tokens"].apply(len)
df[["masked_text", "tokens", "token_count"]]

,masked_text,tokens,token_count
0,اريد المشاركه تواصل على [PHONE] أو [EMAIL],"[ا, ##ريد, ال, ##م, ##شارك, ##ه, تو, ##اصل, عل...",21
1,لم يرضني العمل راسلني على [EMAIL] أو [PHONE],"[لم, ي, ##رض, ##ني, العمل, را, ##سل, ##ني, على...",21
2,اريد المشاركه,"[ا, ##ريد, ال, ##م, ##شارك, ##ه]",6
3,I enjoyed the experince,"[I, enjoyed, the, ex, ##peri, ##nce]",6
4,the work did not satisfy me,"[the, work, did, not, sat, ##is, ##fy, me]",8
5,I would like to participate,"[I, would, like, to, participate]",5


In [ ]:
df["word_count"] = df["masked_text"].apply(
   lambda x: len(x.split())
)

df["fertility"] = df["token_count"] / df["word_count"]

df[["masked_text", "word_count", "token_count", "fertility"]]

,masked_text,word_count,token_count,fertility
0,اريد المشاركه تواصل على [PHONE] أو [EMAIL],7,21,3.000000
1,لم يرضني العمل راسلني على [EMAIL] أو [PHONE],8,21,2.625000
2,اريد المشاركه,2,6,3.000000
3,I enjoyed the experince,4,6,1.500000
4,the work did not satisfy me,6,8,1.333333
5,I would like to participate,5,5,1.000000


In [ ]:
arabic_fertility = df.iloc[:3]["fertility"].mean()
english_fertility = df.iloc[3:]["fertility"].mean()

print("Arabic Fertility:", arabic_fertility)
print("English Fertility:", english_fertility)

Arabic Fertility: 2.875
English Fertility: 1.2777777777777777


In [ ]:
p95_length = df["token_count"].quantile(0.95)

print("p95 Sequence Length:", p95_length)

p95 Sequence Length: 21.0


In [ ]:
unk_token = tokenizer.unk_token

total_tokens = sum(len(tokens) for tokens in df["tokens"])
unk_tokens = sum(tokens.count(unk_token) for tokens in df["tokens"])

unk_rate = unk_tokens / total_tokens

print("UNK Rate:", unk_rate)

UNK Rate: 0.0
